# บทที่ 7 การเรียนรู้แบบไม่มีผู้สอนและการจัดกลุ่ม
โน้ตบุ๊กประกอบตำรา *วิทยาศาสตร์ข้อมูลและการเรียนรู้ของเครื่องสำหรับนักฟิสิกส์*
โค้ดทุกเซลล์คัดลอกมาจากสคริปต์ในโฟลเดอร์ `ch07_clustering/` หัวเซลล์ **โค้ดที่ N.M** ตรงกับเลขในตำรา

**วิธีใช้**
1. รันเซลล์ **เตรียมโค้ด** ด้านล่างก่อนเสมอ (ทุกครั้งที่เปิดโน้ตบุ๊กหรือเริ่ม session ใหม่)
2. แต่ละหัวข้อเริ่มด้วยเซลล์ที่ขึ้นต้นด้วย `%reset -f` ซึ่งล้างตัวแปรเก่า ทำให้ผลเหมือนรันสคริปต์นั้นเดี่ยว ๆ
   ให้รันเซลล์ในหัวข้อเดียวกันเรียงจากบนลงล่าง (Shift+Enter) หรือใช้ Runtime → Run all
3. ใช้หน่วยประมวลผลกลาง (CPU) ตามค่าเริ่มต้น ไม่ต้องเปลี่ยนเป็น GPU
4. ไฟล์ที่สร้างขึ้นอยู่ใน `ch07_clustering/outputs/` และหายเมื่อ session จบ ถ้าต้องการเก็บ ให้คัดลอกไปยัง Google Drive

ตัวเลขส่วนใหญ่ควรตรงกับตำรา การคำนวณที่ไวต่อการปัดเศษ (การฝึกโครงข่ายประสาทเทียม
และมอนติคาร์โลในพิกัดต่อเนื่อง) อาจต่างเล็กน้อยเมื่อรันบนเครื่องอื่น ดู `ch07_clustering/README.md`


In [ ]:
# เตรียมโค้ด: ดึงโค้ดจาก GitHub (เมื่อรันบน Colab) และติดตั้งเฉพาะไลบรารีที่ยังไม่มี
import os, sys, subprocess, importlib.util
CHAPTER = 'ch07_clustering'
REPO = 'https://github.com/busarapa-stack/dsml-for-physicists.git'
if 'google.colab' in sys.modules:
    ROOT = '/content/dsml-for-physicists'
    if not os.path.isdir(ROOT):
        subprocess.run(['git', 'clone', '-q', REPO, ROOT], check=True)
else:                                   # Jupyter บนเครื่องตนเอง: เปิดจากโฟลเดอร์ notebooks/
    ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(os.path.join(ROOT, CHAPTER))
os.makedirs('data', exist_ok=True); os.makedirs('outputs', exist_ok=True)
NEED = {'matplotlib': 'matplotlib', 'numpy': 'numpy', 'pandas': 'pandas', 'scipy': 'scipy', 'sklearn': 'scikit-learn>=1.3'}
missing = [req for mod, req in NEED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
print('โฟลเดอร์ทำงาน:', os.getcwd())
print('ติดตั้งเพิ่ม:', ', '.join(missing) if missing else 'ไม่มี')


## `01_kmeans_elbow.py` — โค้ดที่ 7.1, 7.2

Ch.7  K-Means on three blobs and the elbow method.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 01_kmeans_elbow.py  จากโฟลเดอร์ของบท
__file__ = '01_kmeans_elbow.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.7  K-Means on three blobs and the elbow method.

Book references: sec:kmeans, ssec:choose-k, code:kmeans, code:elbow (line for line)
Data: make_blobs (SYNTHETIC), random_state 2026.
"""
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.chdir(Path(__file__).resolve().parent)
Path('outputs').mkdir(exist_ok=True)

### โค้ดที่ 7.1

In [ ]:
# --- code:kmeans -------------------------------------------------------------
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import make_blobs

X, _ = make_blobs(n_samples=600, centers=3, cluster_std=1.0, random_state=2026)
X_scaled = StandardScaler().fit_transform(X)

pipe = Pipeline([
    ('scaler',  StandardScaler()),
    ('cluster', KMeans(n_clusters=3, init='k-means++', n_init=10,
                       random_state=42)),
])
labels = pipe.fit_predict(X)
centroids = pipe.named_steps['cluster'].cluster_centers_   # in scaled units

### โค้ดที่ 7.2

In [ ]:
# --- code:elbow --------------------------------------------------------------
import numpy as np

wcss = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X_scaled)
    wcss.append(km.inertia_)          # inertia_ is W for this K
# plot range(1, 11) against wcss and look for the elbow

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    print("cluster sizes:", np.bincount(labels).tolist())
    print("centroids (scaled units):\n", centroids.round(2))
    print("W(K) for K = 1..10:", [round(w, 1) for w in wcss])
    print(f"W drops {wcss[0]:.0f} -> {wcss[1]:.0f} -> {wcss[2]:.1f}; "
          f"after K = 3 each extra cluster removes only {wcss[2] - wcss[3]:.1f} to {wcss[8] - wcss[9]:.1f}")

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].scatter(X_scaled[:, 0], X_scaled[:, 1], c=labels, s=8, cmap='viridis')
    axes[0].scatter(centroids[:, 0], centroids[:, 1], c='red', marker='x', s=120)
    axes[0].set_title('K-Means, K = 3 (scaled units)')
    axes[1].plot(range(1, 11), wcss, 'o-'); axes[1].set_xlabel('K'); axes[1].set_ylabel('W (inertia)')
    axes[1].set_title('elbow at K = 3')
    fig.tight_layout(); fig.savefig('outputs/ch07_kmeans_elbow.png', dpi=100)
    print("saved outputs/ch07_kmeans_elbow.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `02_dbscan_hdbscan.py` — โค้ดที่ 7.3, 7.4

Ch.7  DBSCAN and HDBSCAN on two moons; chaining at a larger epsilon.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 02_dbscan_hdbscan.py  จากโฟลเดอร์ของบท
__file__ = '02_dbscan_hdbscan.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.7  DBSCAN and HDBSCAN on two moons; chaining at a larger epsilon.

Book references: sec:dbscan, ssec:dbscan-algo, code:dbscan, code:hdbscan (line for line)
The listings continue the notebook of 01_kmeans_elbow.py (StandardScaler, KMeans), which is executed first.
Data: make_moons (SYNTHETIC), random_state 2026.
"""
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = Path(__file__).resolve().parent
os.chdir(HERE)
Path('outputs').mkdir(exist_ok=True)
_src = (HERE / '01_kmeans_elbow.py').read_text()
exec(_src[_src.index('\n# --- code:'):_src.index('\n# --- end of listings')])   # earlier listings

### โค้ดที่ 7.3

In [ ]:
# --- code:dbscan -------------------------------------------------------------
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_moons

X_moon, y_moon = make_moons(n_samples=500, noise=0.1, random_state=2026)
X_moon = StandardScaler().fit_transform(X_moon)

dbscan = DBSCAN(eps=0.3, min_samples=5)
labels = dbscan.fit_predict(X_moon)          # label -1 means noise
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = (labels == -1).sum()

### โค้ดที่ 7.4

In [ ]:
# --- code:hdbscan ------------------------------------------------------------
from sklearn.cluster import HDBSCAN

clusterer = HDBSCAN(min_cluster_size=15, min_samples=5, copy=True)
labels_h = clusterer.fit_predict(X_moon)
prob_h = clusterer.probabilities_       # strength of membership, 0 for noise

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    from sklearn.metrics import adjusted_rand_score
    print(f"DBSCAN eps=0.3: {n_clusters} clusters, {n_noise} noise point(s), "
          f"ARI vs y_moon = {adjusted_rand_score(y_moon, labels):.3f}")
    lab35 = DBSCAN(eps=0.35, min_samples=5).fit_predict(X_moon)
    print(f"DBSCAN eps=0.35: {len(set(lab35)) - (-1 in lab35)} cluster(s) -> the two moons are chained together")
    km2 = KMeans(n_clusters=2, n_init=10, random_state=42).fit_predict(X_moon)
    print(f"K-Means K=2: ARI = {adjusted_rand_score(y_moon, km2):.3f} (straight cut through both moons)")
    nh = len(set(labels_h)) - (-1 in labels_h)
    print(f"HDBSCAN: {nh} clusters, {(labels_h == -1).sum()} noise points, "
          f"ARI = {adjusted_rand_score(y_moon, labels_h):.3f}, median membership strength {np.median(prob_h[labels_h >= 0]):.2f}")

    fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
    for ax, lab, title in zip(axes, [km2, labels, lab35, labels_h],
                              ['K-Means K=2', 'DBSCAN eps=0.3', 'DBSCAN eps=0.35', 'HDBSCAN']):
        ax.scatter(X_moon[:, 0], X_moon[:, 1], c=lab, s=6, cmap='viridis')
        ax.scatter(*X_moon[lab == -1].T, c='red', marker='x', s=30)
        ax.set_title(title)
    fig.tight_layout(); fig.savefig('outputs/ch07_moons.png', dpi=100)
    print("saved outputs/ch07_moons.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `03_gmm_polymer.py` — โค้ดที่ 7.5

Ch.7  Gaussian mixture for toy polymer conformations (Rg, Ree); responsibilities and BIC.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 03_gmm_polymer.py  จากโฟลเดอร์ของบท
__file__ = '03_gmm_polymer.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.7  Gaussian mixture for toy polymer conformations (Rg, Ree); responsibilities and BIC.

Book references: sec:gmm, ssec:gmm-polymer, ssec:bic, code:gmm-polymer (line for line)
The listing uses numpy from the earlier listings (01_kmeans_elbow.py), which are executed first.
Data: SYNTHETIC three-state chain, seed 2026.
"""
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = Path(__file__).resolve().parent
os.chdir(HERE)
Path('outputs').mkdir(exist_ok=True)
_src = (HERE / '01_kmeans_elbow.py').read_text()
exec(_src[_src.index('\n# --- code:'):_src.index('\n# --- end of listings')])   # earlier listings

### โค้ดที่ 7.5

In [ ]:
# --- code:gmm-polymer --------------------------------------------------------
from sklearn.mixture import GaussianMixture

def make_polymer_states(n=3000, seed=2026):
    """Toy (Rg, Ree) samples of a chain in three metastable states."""
    rng = np.random.default_rng(seed)
    means = np.array([[2.2, 2.5], [3.5, 6.0], [5.0, 12.0]])   # folded, mid, open
    covs  = [np.array([[0.04, 0.03], [0.03, 0.30]]),
             np.array([[0.10, 0.20], [0.20, 1.20]]),
             np.array([[0.30, 0.90], [0.90, 4.00]])]
    pops  = np.array([0.5, 0.2, 0.3])                          # equilibrium weights
    state = rng.choice(3, size=n, p=pops)
    X = np.array([rng.multivariate_normal(means[s], covs[s]) for s in state])
    return X, state

X_polymer, state_true = make_polymer_states()
gmm = GaussianMixture(n_components=3, covariance_type='full',
                      n_init=10, random_state=42).fit(X_polymer)
labels = gmm.predict(X_polymer)
gamma = gmm.predict_proba(X_polymer)            # responsibilities
uncertain = gamma.max(axis=1) < 0.7             # candidate transition frames
print(gmm.weights_, gmm.means_, uncertain.sum())

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    names = ['folded', 'mid', 'open']
    print(f"uncertain frames: {uncertain.sum()} ({100 * uncertain.mean():.1f} %); "
          f"by true state {dict(zip(names, np.bincount(state_true[uncertain], minlength=3).tolist()))}")
    for k in range(3):
        w, v = np.linalg.eigh(gmm.covariances_[k])
        print(f"component {k}: mean {gmm.means_[k].round(2)}, principal eigenvector (Rg, Ree) = "
              f"{np.abs(v[:, -1]).round(3)}, eigenvalues {w.round(3)}")
    bic = [GaussianMixture(n_components=k, covariance_type='full', n_init=10,
                           random_state=42).fit(X_polymer).bic(X_polymer) for k in range(1, 7)]
    print("BIC for K = 1..6:", [round(b) for b in bic], "-> minimum at K =", 1 + int(np.argmin(bic)))

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].scatter(X_polymer[:, 0], X_polymer[:, 1], c=labels, s=5, cmap='viridis')
    axes[0].scatter(*X_polymer[uncertain].T, c='red', s=12, label='max gamma < 0.7')
    axes[0].set_xlabel('R_g'); axes[0].set_ylabel('R_ee'); axes[0].legend()
    axes[1].plot(range(1, 7), bic, 'o-'); axes[1].set_xlabel('K'); axes[1].set_ylabel('BIC')
    fig.tight_layout(); fig.savefig('outputs/ch07_gmm_polymer.png', dpi=100)
    print("saved outputs/ch07_gmm_polymer.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `04_silhouette.py` — โค้ดที่ 7.6

Ch.7  Silhouette score for K = 2..7 on the blobs of code:kmeans.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 04_silhouette.py  จากโฟลเดอร์ของบท
__file__ = '04_silhouette.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.7  Silhouette score for K = 2..7 on the blobs of code:kmeans.

Book references: ssec:silhouette, code:silhouette (line for line)
The listing continues the notebook of 01_kmeans_elbow.py (X_scaled, KMeans, np), which is executed first.
"""
import os
from pathlib import Path

HERE = Path(__file__).resolve().parent
os.chdir(HERE)
_src = (HERE / '01_kmeans_elbow.py').read_text()
exec(_src[_src.index('\n# --- code:'):_src.index('\n# --- end of listings')])   # earlier listings

### โค้ดที่ 7.6

In [ ]:
# --- code:silhouette ---------------------------------------------------------
from sklearn.metrics import silhouette_score, silhouette_samples

sil_scores = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels_k = km.fit_predict(X_scaled)
    sil_scores.append(silhouette_score(X_scaled, labels_k))
best_k = 2 + np.argmax(sil_scores)

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    for k, s in zip(range(2, 8), sil_scores):
        print(f"K = {k}: silhouette {s:.3f}")
    print("best K =", best_k)

## `05_afm_three_algorithms.py` — โค้ดที่ 7.7

Ch.7  Toy AFM force map: K-Means, GMM and Ward (subsample + nearest centroid), k = 2..5,

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 05_afm_three_algorithms.py  จากโฟลเดอร์ของบท
__file__ = '05_afm_three_algorithms.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.7  Toy AFM force map: K-Means, GMM and Ward (subsample + nearest centroid), k = 2..5,
winner-takes-all and soft rules, F1 / MCC against the reference map.

Book references: sec:afm-case, ssec:afm-PI, code:afm-3algo (line for line), tab:afm-results
The map is SYNTHETIC (seed 2026).  It imitates the design of the author's AFM study; no unpublished data are used.
The listing continues the notebook: KMeans / np (01), GaussianMixture (03), silhouette_score (04) are executed first.
"""
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = Path(__file__).resolve().parent
os.chdir(HERE)
Path('outputs').mkdir(exist_ok=True)
for _f in ['01_kmeans_elbow.py', '03_gmm_polymer.py', '04_silhouette.py']:
    _src = (HERE / _f).read_text()
    exec(_src[_src.index('\n# --- code:'):_src.index('\n# --- end of listings')])   # earlier listings

### โค้ดที่ 7.7

In [ ]:
# --- code:afm-3algo ----------------------------------------------------------
from scipy.ndimage import gaussian_filter
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.neighbors import NearestCentroid
from sklearn.metrics import f1_score, matthews_corrcoef

def make_scaffold_map(n=128, seed=2026):
    """Toy AFM force map: scaffold (label 1) vs substrate (paraffin or glass).
    Channels: height H (nm), contact stiffness S (N/m), adhesion A (nN)."""
    rng = np.random.default_rng(seed)
    phi = gaussian_filter(rng.normal(size=(n, n)), sigma=6); phi /= phi.std()
    scaffold = phi > -0.55                                    # about 70 % scaffold
    g = gaussian_filter(rng.normal(size=(n, n)), sigma=4); g /= g.std()
    glass = (~scaffold) & (g > 0.8)                           # exposed glass
    het = gaussian_filter(rng.normal(size=(n, n)), sigma=3); het /= het.std()
    H = np.where(scaffold, 250 + 72 * het, np.where(glass, 20.0, 130.0))
    S = np.where(scaffold, 0.8 + 0.18 * het, np.where(glass, 1.5, 0.5))
    A = np.where(scaffold, 4.5 - 0.72 * het, np.where(glass, 2.5, 6.5))
    H, S, A = (gaussian_filter(m, sigma=1.0) for m in (H, S, A))   # tip blur
    heavy = lambda s: s * rng.standard_t(df=3, size=(n, n))        # heavy tails
    H += heavy(30.0); S += heavy(0.09); A += heavy(0.65)
    X = np.column_stack([H.ravel(), S.ravel(), A.ravel()])
    return X, scaffold.ravel().astype(int), glass.ravel()

X_afm, y_afm, glass_afm = make_scaffold_map()
Xa = RobustScaler().fit_transform(X_afm)

def ward_labels(k, n_sub=5000, seed=42):
    """Ward on a subsample, then assign every pixel to the nearest centroid."""
    idx = np.random.default_rng(seed).choice(len(Xa), n_sub, replace=False)
    lab = AgglomerativeClustering(n_clusters=k, linkage='ward').fit_predict(Xa[idx])
    return NearestCentroid().fit(Xa[idx], lab).predict(Xa)

algorithms = {
    'K-Means': lambda k: KMeans(k, n_init=20, random_state=42).fit_predict(Xa),
    'GMM':     lambda k: GaussianMixture(k, covariance_type='full', n_init=10,
                                         random_state=42).fit(Xa).predict(Xa),
    'Ward':    ward_labels,
}
sub = np.random.default_rng(0).choice(len(Xa), 5000, replace=False)
for name, fit in algorithms.items():
    for k in range(2, 6):
        lab = fit(k)
        share = {c: y_afm[lab == c].mean() for c in np.unique(lab)}  # evaluation only
        wta  = (lab == max(share, key=share.get)).astype(int)
        soft = np.isin(lab, [c for c, v in share.items() if v >= 0.5]).astype(int)
        print(f"{name:8s} k={k} sil={silhouette_score(Xa[sub], lab[sub]):.3f} "
              f"WTA F1={f1_score(y_afm, wta):.3f} MCC={matthews_corrcoef(y_afm, wta):.3f} "
              f"soft F1={f1_score(y_afm, soft):.3f} MCC={matthews_corrcoef(y_afm, soft):.3f}")

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    from sklearn.cluster import DBSCAN
    from sklearn.metrics import davies_bouldin_score, calinski_harabasz_score
    print(f"\nscaffold fraction {y_afm.mean():.3f}, exposed glass {glass_afm.mean():.3f}, "
          f"paraffin {1 - y_afm.mean() - glass_afm.mean():.3f}")
    for j, ch in enumerate('HSA'):
        dev = np.abs(X_afm[:, j] - np.median(X_afm[:, j]))
        print(f"{ch}: 99.9th percentile of |x - median| / median of |x - median| = "
              f"{np.percentile(dev, 99.9) / np.median(dev):.1f}  (Gaussian: 4.9)")

    print("\nDBSCAN on the full scaled map (min_samples=5):")
    for eps in (0.3, 0.5):
        lab = DBSCAN(eps=eps, min_samples=5).fit_predict(Xa)
        sizes = np.sort(np.bincount(lab[lab >= 0]))[::-1]
        print(f"  eps={eps}: largest cluster {sizes[0]} of {len(Xa)} pixels ({100 * sizes[0] / len(Xa):.1f} %), "
              f"other clusters have at most {sizes[1] if len(sizes) > 1 else 0} pixels, noise {(lab == -1).sum()}")

    print("\nK-Means clusters in physical units (median H nm, S N/m, A nN):")
    for k in (2, 3):
        lab = KMeans(k, n_init=20, random_state=42).fit_predict(Xa)
        for c in np.argsort([np.median(X_afm[lab == c, 0]) for c in range(k)]):
            m = lab == c
            print(f"  k={k} cluster: {100 * m.mean():4.1f} % of pixels, scaffold {100 * y_afm[m].mean():4.1f} %, "
                  f"glass {100 * glass_afm[m].mean():4.1f} %, median (H, S, A) = {np.median(X_afm[m], axis=0).round(2)}")

    print("\ninternal indices (K-Means; GMM for BIC):")
    for k in range(2, 7):
        lab = KMeans(k, n_init=20, random_state=42).fit_predict(Xa)
        bic = GaussianMixture(k, covariance_type='full', n_init=10, random_state=42).fit(Xa).bic(Xa)
        print(f"  k={k} silhouette {silhouette_score(Xa[sub], lab[sub]):.3f}  "
              f"Davies-Bouldin {davies_bouldin_score(Xa, lab):.3f}  "
              f"Calinski-Harabasz {calinski_harabasz_score(Xa, lab):.0f}  GMM BIC {bic:.0f}")

    lab3 = KMeans(3, n_init=20, random_state=42).fit_predict(Xa)
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    for ax, img, title in zip(axes, [X_afm[:, 0], X_afm[:, 1], X_afm[:, 2], y_afm + glass_afm * 0.5, lab3],
                              ['H (nm)', 'S (N/m)', 'A (nN)', 'reference (1 scaffold, 0.5 glass)', 'K-Means k=3']):
        img = np.asarray(img, float)
        lo, hi = np.percentile(img, [1, 99])      # heavy tails would wash out the colour scale
        im = ax.imshow(img.reshape(128, 128), cmap='viridis', vmin=lo, vmax=hi); ax.set_title(title); ax.axis('off')
        fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout(); fig.savefig('outputs/ch07_afm_maps.png', dpi=90)
    print("saved outputs/ch07_afm_maps.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


# แนวคำตอบของแบบฝึกหัดที่ต้องรันโค้ด
ควรลองทำเองก่อน แล้วจึงรันเซลล์เหล่านี้เพื่อเทียบคำตอบ

## `exercises/E7_2_elbow_spread.py` — โค้ดที่ 7.1

E7.2  Elbow and silhouette for cluster_std = 1.0 (code:kmeans) and 2.5.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E7_2_elbow_spread.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E7_2_elbow_spread.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E7.2  Elbow and silhouette for cluster_std = 1.0 (code:kmeans) and 2.5."""
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

HERE = Path(__file__).resolve().parent.parent
os.chdir(HERE); Path('outputs').mkdir(exist_ok=True)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for row, std in enumerate([1.0, 2.5]):
    X, _ = make_blobs(n_samples=600, centers=3, cluster_std=std, random_state=2026)
    X_scaled = StandardScaler().fit_transform(X)
    W = [KMeans(k, n_init=10, random_state=42).fit(X_scaled).inertia_ for k in range(1, 11)]
    sil = [silhouette_score(X_scaled, KMeans(k, n_init=10, random_state=42).fit_predict(X_scaled))
           for k in range(2, 8)]
    k_star = 2 + int(np.argmax(sil))
    print(f"cluster_std = {std}: W(K=1..6) = {[round(w) for w in W[:6]]}")
    print(f"   silhouette K=2..7 = {[round(s, 3) for s in sil]} -> K* = {k_star}")
    km = KMeans(k_star, n_init=10, random_state=42).fit(X_scaled)
    axes[row, 0].plot(range(1, 11), W, 'o-'); axes[row, 0].set_title(f'std {std}: W vs K')
    axes[row, 1].plot(range(2, 8), sil, 'o-'); axes[row, 1].set_title('silhouette vs K')
    axes[row, 2].scatter(*X_scaled.T, c=km.labels_, s=6); axes[row, 2].scatter(*km.cluster_centers_.T, c='red', marker='x', s=100)
    axes[row, 2].set_title(f'K* = {k_star}')
fig.tight_layout(); fig.savefig('outputs/E7_2_elbow_spread.png', dpi=90)
print("saved outputs/E7_2_elbow_spread.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `exercises/E7_3_moons_dbscan.py` — โค้ดที่ 7.3

E7.3  K-Means vs DBSCAN (eps 0.2-0.35) on the moons of code:dbscan: ARI, silhouette, k-distance plot.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E7_3_moons_dbscan.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E7_3_moons_dbscan.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E7.3  K-Means vs DBSCAN (eps 0.2-0.35) on the moons of code:dbscan: ARI, silhouette, k-distance plot."""
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import DBSCAN, KMeans
from sklearn.datasets import make_moons
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

HERE = Path(__file__).resolve().parent.parent
os.chdir(HERE); Path('outputs').mkdir(exist_ok=True)

X_moon, y_moon = make_moons(n_samples=500, noise=0.1, random_state=2026)   # as in code:dbscan
X_moon = StandardScaler().fit_transform(X_moon)

km = KMeans(2, n_init=10, random_state=42).fit_predict(X_moon)
print(f"K-Means K=2    : ARI {adjusted_rand_score(y_moon, km):.3f}  silhouette {silhouette_score(X_moon, km):.3f}")
fig, axes = plt.subplots(1, 6, figsize=(22, 3.6))
axes[0].scatter(*X_moon.T, c=km, s=5); axes[0].set_title('K-Means')
for ax, eps in zip(axes[1:5], [0.2, 0.25, 0.3, 0.35]):
    lab = DBSCAN(eps=eps, min_samples=5).fit_predict(X_moon)
    nc = len(set(lab)) - (-1 in lab)
    ok = lab >= 0
    sil = f"{silhouette_score(X_moon[ok], lab[ok]):.3f}" if nc > 1 else "  -  "
    print(f"DBSCAN eps={eps:<4}: {nc} cluster(s), noise {100 * (~ok).mean():.1f} %, "
          f"ARI {adjusted_rand_score(y_moon, lab):.3f}  silhouette (noise excluded) {sil}")
    ax.scatter(*X_moon.T, c=lab, s=5); ax.set_title(f'DBSCAN eps={eps}')
kd = np.sort(NearestNeighbors(n_neighbors=6).fit(X_moon).kneighbors(X_moon)[0][:, 5])  # 5th neighbour (self excluded)
print(f"5th-neighbour distance: median {np.median(kd):.2f}, 95th pct {np.percentile(kd, 95):.2f}, "
      f"99th pct {np.percentile(kd, 99):.2f}, max {kd.max():.2f}")
axes[5].plot(kd); axes[5].set_title('k-distance (k = 5)'); axes[5].axhline(0.3, ls='--', c='gray')
fig.tight_layout(); fig.savefig('outputs/E7_3_moons.png', dpi=90)
print("saved outputs/E7_3_moons.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `exercises/E7_4_gmm_bic_covtypes.py` — โค้ดที่ 7.5

E7.4  BIC for K = 1..6 and the four covariance types at K*, compared with K-Means (polymer data of code:gmm-polymer).

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E7_4_gmm_bic_covtypes.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E7_4_gmm_bic_covtypes.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E7.4  BIC for K = 1..6 and the four covariance types at K*, compared with K-Means (polymer data of code:gmm-polymer)."""
import os
import sys
from pathlib import Path

import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

HERE = Path(__file__).resolve().parent.parent
os.chdir(HERE)
__file__ = str(HERE / '03_gmm_polymer.py')
src = (HERE / '03_gmm_polymer.py').read_text()
exec(src[:src.index('\n# --- end of listings')].split('"""', 2)[2])     # builds X_polymer, state_true

bic = [GaussianMixture(k, covariance_type='full', n_init=10, random_state=42).fit(X_polymer).bic(X_polymer)
       for k in range(1, 7)]
k_star = 1 + int(np.argmin(bic))
print("BIC K=1..6:", [round(b) for b in bic], "-> K* =", k_star)
print("true state fractions:", np.bincount(state_true).round() / len(state_true))
for ct in ['full', 'diag', 'tied', 'spherical']:
    m = GaussianMixture(k_star, covariance_type=ct, n_init=10, random_state=42).fit(X_polymer)
    print(f"{ct:9s}: BIC {m.bic(X_polymer):7.0f}  ARI {adjusted_rand_score(state_true, m.predict(X_polymer)):.3f}  "
          f"weights {np.sort(m.weights_).round(3)}")
lab = KMeans(3, n_init=10, random_state=42).fit_predict(StandardScaler().fit_transform(X_polymer))
print(f"K-Means  : ARI {adjusted_rand_score(state_true, lab):.3f}  cluster fractions {np.sort(np.bincount(lab) / len(lab)).round(3)}")

## `exercises/E7_5_afm_protocol.py` — โค้ดที่ 7.7

E7.5  AFM protocol: choose k from internal indices only, then evaluate (F1, MCC, balanced accuracy, both rules),

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E7_5_afm_protocol.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E7_5_afm_protocol.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E7.5  AFM protocol: choose k from internal indices only, then evaluate (F1, MCC, balanced accuracy, both rules),
interpret clusters in physical units, and compare feature subsets.  SYNTHETIC map from code:afm-3algo."""
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = Path(__file__).resolve().parent.parent
os.chdir(HERE); Path('outputs').mkdir(exist_ok=True)
for _f in ['01_kmeans_elbow.py', '03_gmm_polymer.py', '04_silhouette.py']:
    _s = (HERE / _f).read_text(); exec(_s[_s.index('\n# --- code:'):_s.index('\n# --- end of listings')])
_s = (HERE / '05_afm_three_algorithms.py').read_text()
_s = _s[_s.index('\n# --- code:afm'):_s.index('\n# --- end of listings')]
exec(_s[:_s.index('sub = np.random')])            # data, scaler and the three algorithms, without the printing loop

from sklearn.metrics import balanced_accuracy_score, calinski_harabasz_score, davies_bouldin_score

sub = np.random.default_rng(0).choice(len(Xa), 5000, replace=False)
rules = {}
print("step 1: internal indices only (reference map not used)")
choice = {}
for name, fit in algorithms.items():
    rows = []
    for k in range(2, 6):
        lab = fit(k)
        rows.append((k, silhouette_score(Xa[sub], lab[sub]), davies_bouldin_score(Xa, lab), calinski_harabasz_score(Xa, lab), lab))
    for k, s, d, c, _ in rows:
        print(f"  {name:8s} k={k}: silhouette {s:.3f}  DB {d:.3f}  CH {c:.0f}")
    k_sil = max(rows, key=lambda r: r[1])[0]; k_db = min(rows, key=lambda r: r[2])[0]; k_ch = max(rows, key=lambda r: r[3])[0]
    print(f"  {name}: silhouette -> k={k_sil}, Davies-Bouldin -> k={k_db}, Calinski-Harabasz -> k={k_ch}; chosen k = {k_sil} (silhouette)")
    choice[name] = [r for r in rows if r[0] == k_sil][0][4]


def to_binary(lab):
    share = {c: y_afm[lab == c].mean() for c in np.unique(lab)}
    wta = (lab == max(share, key=share.get)).astype(int)
    soft = np.isin(lab, [c for c, v in share.items() if v >= 0.5]).astype(int)
    return wta, soft


print("\nstep 2: evaluation of the chosen k against the reference map")
base = np.ones_like(y_afm)
print(f"  all-scaffold baseline: F1 {f1_score(y_afm, base):.3f}  MCC {matthews_corrcoef(y_afm, base):.3f}  "
      f"balanced acc {balanced_accuracy_score(y_afm, base):.3f}")
for name, lab in choice.items():
    for rule, pred in zip(['WTA', 'soft'], to_binary(lab)):
        print(f"  {name:8s} {rule:4s}: F1 {f1_score(y_afm, pred):.3f}  MCC {matthews_corrcoef(y_afm, pred):.3f}  "
              f"balanced acc {balanced_accuracy_score(y_afm, pred):.3f}")

print("\nstep 3: K-Means k=3 clusters in physical units (median H nm, S N/m, A nN)")
lab = choice['K-Means']
for c in np.unique(lab):
    m = lab == c
    print(f"  cluster {c}: {100 * m.mean():4.1f} %  scaffold {100 * y_afm[m].mean():4.1f} %  glass {100 * glass_afm[m].mean():4.1f} %  "
          f"median {np.median(X_afm[m], axis=0).round(2)}")

print("\nstep 4: feature subsets (K-Means, k = 3, RobustScaler on the subset)")
for fs in ['H', 'S', 'A', 'HA', 'HSA']:
    Z = RobustScaler().fit_transform(X_afm[:, ['HSA'.index(c) for c in fs]])
    lab = KMeans(3, n_init=20, random_state=42).fit_predict(Z)
    wta, soft = to_binary(lab)
    print(f"  {fs:3s}: MCC WTA {matthews_corrcoef(y_afm, wta):.3f}  soft {matthews_corrcoef(y_afm, soft):.3f}")

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
imgs = [(y_afm + 0.5 * glass_afm, 'reference')] + [(algorithms[n](k), f'{n} k={k}') for k in (2, 3) for n in algorithms]
for ax, (img, t) in zip(axes.ravel(), imgs):
    ax.imshow(np.asarray(img, float).reshape(128, 128)); ax.set_title(t); ax.axis('off')
axes.ravel()[-1].axis('off')
fig.tight_layout(); fig.savefig('outputs/E7_5_afm_maps.png', dpi=80)
print("saved outputs/E7_5_afm_maps.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `exercises/E7_6_sdss_unsupervised.py`

E7.6  Cluster SDSS objects on four colour indices only (no redshift, no labels), then compare with the labels.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E7_6_sdss_unsupervised.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E7_6_sdss_unsupervised.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E7.6  Cluster SDSS objects on four colour indices only (no redshift, no labels), then compare with the labels.

Uses the SDSS table of Chapter 6: ../ch06_classification/data/raw/star_classification.csv
(the real Kaggle file if the reader put it there, otherwise ch06's SYNTHETIC stand-in is created).
"""
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

HERE = Path(__file__).resolve().parent.parent
os.chdir(HERE)
CH6 = HERE.parent / 'ch06_classification'
path = CH6 / 'data/raw/star_classification.csv'
if not path.exists():
    subprocess.run([sys.executable, str(CH6 / '05_sdss_data.py')], check=True)
print((CH6 / 'data/raw/SOURCE.txt').read_text().strip())

df = pd.read_csv(path)
bands = ['u', 'g', 'r', 'i', 'z']
df = df[(df[bands] > -100).all(axis=1)]
C = np.column_stack([df.u - df.g, df.g - df.r, df.r - df.i, df.i - df.z])   # colour indices as in code:sdss-cc
y = df['class'].to_numpy()
Z = StandardScaler().fit_transform(C)

km = KMeans(3, n_init=10, random_state=42).fit_predict(Z)
print(f"\nK-Means K=3: ARI {adjusted_rand_score(y, km):.3f}")
print(pd.crosstab(km, y))

sub = np.random.default_rng(0).choice(len(Z), 20000, replace=False)     # BIC on a 20,000-object subsample
bic = [GaussianMixture(k, covariance_type='full', n_init=3, random_state=42).fit(Z[sub]).bic(Z[sub]) for k in range(1, 11)]
k_star = 1 + int(np.argmin(bic))
print("\nGMM BIC K=1..10:", [round(b) for b in bic], "-> K* =", k_star)
gm = GaussianMixture(k_star, covariance_type='full', n_init=3, random_state=42).fit(Z[sub]).predict(Z)
print(f"GMM K={k_star}: ARI {adjusted_rand_score(y, gm):.3f}")
tab = pd.crosstab(gm, y)
tab['u-g'] = pd.Series(C[:, 0]).groupby(gm).median().round(2)
tab['g-r'] = pd.Series(C[:, 1]).groupby(gm).median().round(2)
print(tab)
q = tab['QSO'] / tab[['GALAXY', 'QSO', 'STAR']].sum(axis=1)
best = q.idxmax()
print(f"most quasar-rich GMM cluster: {100 * q[best]:.0f} % QSO, holds {100 * tab.loc[best, 'QSO'] / (y == 'QSO').sum():.0f} % of all QSOs, "
      f"median u-g {tab.loc[best, 'u-g']}")

## `exercises/E7_7_bootstrap_stability.py` — โค้ดที่ 7.7

E7.7  Bootstrap stability of K-Means (k = 2..5) and GMM (k = 3) on the SYNTHETIC AFM map of code:afm-3algo.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E7_7_bootstrap_stability.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E7_7_bootstrap_stability.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E7.7  Bootstrap stability of K-Means (k = 2..5) and GMM (k = 3) on the SYNTHETIC AFM map of code:afm-3algo.

Data resampling and algorithm initialisation use different seeds (DATA_SEED vs 1000 + b).
    python E7_7_bootstrap_stability.py            # DATA_SEED = 2026, about 2 minutes
    python E7_7_bootstrap_stability.py --seeds    # repeat for data seeds 0, 1, 2026 (about 6 minutes)
"""
import os
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = Path(__file__).resolve().parent.parent
os.chdir(HERE); Path('outputs').mkdir(exist_ok=True)
for _f in ['01_kmeans_elbow.py', '03_gmm_polymer.py', '04_silhouette.py']:
    _s = (HERE / _f).read_text(); exec(_s[_s.index('\n# --- code:'):_s.index('\n# --- end of listings')])
_s = (HERE / '05_afm_three_algorithms.py').read_text()
_s = _s[_s.index('\n# --- code:afm'):_s.index('\n# --- end of listings')]
exec(_s[:_s.index('sub = np.random')])

from scipy.optimize import linear_sum_assignment
from sklearn.metrics import adjusted_rand_score, confusion_matrix

N_BOOT = 20


def align(ref, lab, k):
    """Relabel lab so that its clusters match ref as well as possible (Hungarian matching)."""
    C = confusion_matrix(ref, lab, labels=range(k))
    r, c = linear_sum_assignment(-C)
    return np.array([dict(zip(c, r))[v] for v in range(k)])[lab]


def model(name, k, seed):
    if name == 'K-Means':
        return KMeans(k, n_init=20, random_state=seed)
    return GaussianMixture(k, covariance_type='full', n_init=10, random_state=seed)


def stability(data_seed, show=False):
    rng = np.random.default_rng(data_seed)
    boots = [rng.choice(len(Xa), len(Xa), replace=True) for _ in range(N_BOOT)]
    unstable_maps = {}
    for name, k in [('K-Means', 2), ('K-Means', 3), ('K-Means', 4), ('K-Means', 5), ('GMM', 3)]:
        ref = model(name, k, 42).fit(Xa).predict(Xa)
        ari, same = [], np.zeros(len(Xa))
        for b, idx in enumerate(boots):
            lab = model(name, k, 1000 + b).fit(Xa[idx]).predict(Xa)
            ari.append(adjusted_rand_score(ref, lab))
            same += align(ref, lab, k) == ref
        unstable = same < 18
        unstable_maps[(name, k)] = (unstable, ref)
        print(f"  data seed {data_seed:4d}  {name:7s} k={k}: ARI mean {np.mean(ari):.3f}  min {np.min(ari):.3f}  "
              f"unstable pixels {100 * unstable.mean():.1f} %", flush=True)
    return unstable_maps


maps = stability(2026)
un, ref = maps[('K-Means', 3)]
yy = y_afm.reshape(128, 128)
edge = np.zeros_like(yy, bool)
for sh in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
    edge |= np.roll(yy, sh, (0, 1)) != yy
edge = edge.ravel()
order = np.argsort([np.median(X_afm[ref == c, 0]) for c in range(3)])
names = dict(zip(order, ['glass-like (low)', 'low mixed', 'high scaffold']))
print(f"\nK-Means k=3 unstable pixels: {un.sum()}; on a scaffold/substrate edge in the image: {100 * edge[un].mean():.0f} % "
      f"(all pixels: {100 * edge.mean():.0f} %)")
print("  reference cluster of the unstable pixels:", {names[c]: int((ref[un] == c).sum()) for c in range(3)})
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(y_afm.reshape(128, 128)); ax[0].set_title('reference')
ax[1].imshow(un.reshape(128, 128), cmap='Reds'); ax[1].set_title('K-Means k=3: same cluster < 18/20')
for a in ax: a.axis('off')
fig.tight_layout(); fig.savefig('outputs/E7_7_unstable_pixels.png', dpi=90)
print("saved outputs/E7_7_unstable_pixels.png")

if '--seeds' in sys.argv:
    for s in (0, 1):
        stability(s)

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


# ข้อมูลจริง (ไม่บังคับ)
สคริปต์สำหรับดาวน์โหลดและอ่านข้อมูลจริงอยู่ใน `ch07_clustering/optional/` วิธีดาวน์โหลดและผลที่ควรได้อยู่ใน `ch07_clustering/README.md` ข้อมูลเหล่านี้ไม่ได้อยู่ใน GitHub เพราะขนาดใหญ่หรือมีเงื่อนไขสัญญาอนุญาต